In [6]:
%%writefile device_query.cu
#include <cuda_runtime.h>
#include <stdio.h>

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    printf("Number of CUDA devices: %d\n\n", deviceCount);

    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);

        printf("========== Device %d ==========\n", i);
        printf("Name:                    %s\n", prop.name);
        printf("Compute Capability:      %d.%d\n", prop.major, prop.minor);

        printf("\n--- Block & Grid ---\n");
        printf("Max Threads per Block:   %d\n", prop.maxThreadsPerBlock);
        printf("Max Block Dim (x,y,z):   %d x %d x %d\n",
               prop.maxThreadsDim[0],
               prop.maxThreadsDim[1],
               prop.maxThreadsDim[2]);
        printf("Max Grid Dim  (x,y,z):   %d x %d x %d\n",
               prop.maxGridSize[0],
               prop.maxGridSize[1],
               prop.maxGridSize[2]);
        printf("Warp Size:               %d\n", prop.warpSize);
        printf("Multiprocessors:         %d\n", prop.multiProcessorCount);
        printf("Max Threads per SM:      %d\n", prop.maxThreadsPerMultiProcessor);

        printf("\n--- Memory ---\n");
        printf("Global Memory:           %.2f GB\n",
               prop.totalGlobalMem / (1024.0*1024*1024));
        printf("Shared Memory/Block:     %zu KB\n",
               prop.sharedMemPerBlock / 1024);
        printf("Constant Memory:         %zu KB\n",
               prop.totalConstMem / 1024);
        printf("Registers per Block:     %d\n", prop.regsPerBlock);
        printf("L2 Cache Size:           %d KB\n", prop.l2CacheSize/1024);
        printf("Memory Clock Rate:       %d MHz\n", prop.memoryClockRate/1000);
        printf("Memory Bus Width:        %d bit\n", prop.memoryBusWidth);

        printf("\n--- Other ---\n");
        printf("Clock Rate:              %d MHz\n", prop.clockRate/1000);
        printf("Double Precision:        %s\n",
               prop.major >= 2 ? "Yes" : "No");
        printf("ECC Enabled:             %s\n",
               prop.ECCEnabled ? "Yes" : "No");
        printf("Concurrent Kernels:      %s\n",
               prop.concurrentKernels ? "Yes" : "No");

        printf("================================\n\n");
    }
    return 0;
}

Overwriting device_query.cu


In [7]:
!nvcc device_query.cu -o device_query && ./device_query


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Number of CUDA devices: 1

========== Device 0 ==========
Name:                    Tesla T4
Compute Capability:      7.5

--- Block & Grid ---
Max Threads per Block:   1024
Max Block Dim (x,y,z):   1024 x 1024 x 64
Max Grid Dim  (x,y,z):   2147483647 x 65535 x 65535
Warp Size:               32
Multiprocessors:         40
Max Threads per SM:      1024

--- Memory ---
Global Memory:           14.56 GB
Shared Memory/Block:     48 KB
Constant Memory:         64 KB
Registers per Block:     65536
L2 Cache Size:           4096 KB
Memory Clock Rate:       5001 MHz
Memory Bus Width:        256 bit

--- Other ---
Clock Rate:              1590 MHz
Double Precision:        Yes
ECC Enabled:             Yes
Concurrent Kernels:      Yes



In [8]:
%%writefile array_sum.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>

// Array size = 2^22 = ~4 million elements
#define N (1 << 22)
#define BLOCK_SIZE 256

// ─── GPU Kernel ───────────────────────────────────────────
// Each block reduces its chunk into one value using shared memory
__global__ void sumKernel(float *input, float *blockSums, int n) {
    __shared__ float sdata[BLOCK_SIZE];

    int tid  = threadIdx.x;
    int gid  = blockIdx.x * blockDim.x + threadIdx.x;

    // Load into shared memory (0 if out of bounds)
    sdata[tid] = (gid < n) ? input[gid] : 0.0f;
    __syncthreads();

    // Tree reduction inside the block
    // Each round, half the threads add the other half
    for (int stride = BLOCK_SIZE / 2; stride > 0; stride >>= 1) {
        if (tid < stride) {
            sdata[tid] += sdata[tid + stride];
        }
        __syncthreads();
    }

    // Thread 0 of each block saves the block's partial sum
    if (tid == 0) {
        blockSums[blockIdx.x] = sdata[0];
    }
}

// ─── CPU Reference ────────────────────────────────────────
float cpuSum(float *arr, int n) {
    float sum = 0.0f;
    for (int i = 0; i < n; i++) sum += arr[i];
    return sum;
}

int main() {
    printf("========== Array Sum ==========\n");
    printf("Array Size: %d elements\n\n", N);

    // ── Allocate host array ──
    float *h_input = (float*)malloc(N * sizeof(float));

    // Fill with random floats between 0 and 1
    srand(42);
    for (int i = 0; i < N; i++) {
        h_input[i] = (float)rand() / RAND_MAX;
    }

    // ── CPU Sum ──
    clock_t cpu_start = clock();
    float cpu_result = cpuSum(h_input, N);
    clock_t cpu_end = clock();
    float cpu_time = (float)(cpu_end - cpu_start)
                     / CLOCKS_PER_SEC * 1000.0f;

    printf("CPU Result: %.4f\n", cpu_result);
    printf("CPU Time:   %.4f ms\n\n", cpu_time);

    // ── GPU Setup ──
    int numBlocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;

    // Step 1: Allocate device memory
    float *d_input, *d_blockSums;
    cudaMalloc((void**)&d_input,     N * sizeof(float));
    cudaMalloc((void**)&d_blockSums, numBlocks * sizeof(float));

    // Step 2: Copy host to device
    cudaMemcpy(d_input, h_input,
               N * sizeof(float),
               cudaMemcpyHostToDevice);

    // Step 3: Grid and Block dimensions
    dim3 blockDim(BLOCK_SIZE);
    dim3 gridDim(numBlocks);

    printf("Grid  Size: %d blocks\n", numBlocks);
    printf("Block Size: %d threads\n\n", BLOCK_SIZE);

    // GPU Timing using CUDA Events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    // Step 4: Launch kernel
    sumKernel<<<gridDim, blockDim>>>(d_input, d_blockSums, N);
    cudaDeviceSynchronize();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // Step 5: Copy result back to host
    float *h_blockSums = (float*)malloc(numBlocks * sizeof(float));
    cudaMemcpy(h_blockSums, d_blockSums,
               numBlocks * sizeof(float),
               cudaMemcpyDeviceToHost);

    // Final reduction on CPU
    float gpu_result = 0.0f;
    for (int i = 0; i < numBlocks; i++) {
        gpu_result += h_blockSums[i];
    }

    printf("GPU Result: %.4f\n", gpu_result);
    printf("GPU Time:   %.4f ms\n\n", gpu_time);

    // Verify
    float diff = fabs(cpu_result - gpu_result);
    printf("Difference: %.6f\n", diff);
    printf("Result:     %s\n\n",
           diff < 1.0f ? "MATCH ✓" : "MISMATCH ✗");

    printf("Speedup:    %.2fx\n", cpu_time / gpu_time);

    // Step 6: Free device memory
    cudaFree(d_input);
    cudaFree(d_blockSums);
    free(h_input);
    free(h_blockSums);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing array_sum.cu


In [9]:
!nvcc array_sum.cu -o array_sum && ./array_sum

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
========== Array Sum ==========
Array Size: 4194304 elements

CPU Result: 2097117.6250
CPU Time:   12.7170 ms

Grid  Size: 16384 blocks
Block Size: 256 threads

GPU Result: 2097113.7500
GPU Time:   99.3542 ms

Difference: 3.875000
Result:     MISMATCH ✗

Speedup:    0.13x


In [10]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h>
#include <time.h>

// Matrix is N x N
#define N 2048
#define BLOCK_SIZE 32

// ─── GPU Kernel ───────────────────────────────────────────
// Each thread computes exactly ONE element of C
// C[row][col] = A[row][col] + B[row][col]
__global__ void matrixAddKernel(int *A, int *B, int *C, int width) {
    // Which row and column does THIS thread handle?
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < width && col < width) {
        int idx = row * width + col;
        // 2 global reads + 1 global write
        C[idx] = A[idx] + B[idx];
    }
}

// ─── CPU Reference ────────────────────────────────────────
void cpuMatrixAdd(int *A, int *B, int *C, int n) {
    for (int i = 0; i < n * n; i++) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    printf("========== Matrix Addition ==========\n");
    printf("Matrix Size: %d x %d\n\n", N, N);

    int size = N * N * sizeof(int);

    // Allocate host matrices
    int *h_A    = (int*)malloc(size);
    int *h_B    = (int*)malloc(size);
    int *h_C    = (int*)malloc(size);   // GPU result
    int *h_Cref = (int*)malloc(size);   // CPU reference

    // Fill A and B with random values
    srand(42);
    for (int i = 0; i < N * N; i++) {
        h_A[i] = rand() % 1000;
        h_B[i] = rand() % 1000;
    }

    // ── CPU Addition ──
    clock_t cpu_start = clock();
    cpuMatrixAdd(h_A, h_B, h_Cref, N);
    clock_t cpu_end = clock();
    float cpu_time = (float)(cpu_end - cpu_start)
                     / CLOCKS_PER_SEC * 1000.0f;

    printf("CPU Time: %.4f ms\n", cpu_time);

    // ── GPU Setup ──

    // Allocate device matrices
    int *d_A, *d_B, *d_C;
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy A and B to device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // 2D Block: 32x32 = 1024 threads per block
    dim3 blockDim(BLOCK_SIZE, BLOCK_SIZE);

    // Grid covers the entire N x N matrix
    dim3 gridDim((N + BLOCK_SIZE - 1) / BLOCK_SIZE,
                 (N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    printf("Block Dim: %d x %d = %d threads/block\n",
           BLOCK_SIZE, BLOCK_SIZE, BLOCK_SIZE*BLOCK_SIZE);
    printf("Grid  Dim: %d x %d = %d blocks\n\n",
           gridDim.x, gridDim.y, gridDim.x * gridDim.y);

    // GPU Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);

    // Launch kernel
    matrixAddKernel<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, start, stop);

    // Copy result back
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("GPU Time: %.4f ms\n\n", gpu_time);

    // ── Verify ──
    int errors = 0;
    for (int i = 0; i < N * N; i++) {
        if (h_C[i] != h_Cref[i]) errors++;
    }
    printf("Verification: %s (errors: %d)\n\n",
           errors == 0 ? "PASS ✓" : "FAIL ✗", errors);

    // ── Analysis ──
    long long total     = (long long)N * N;
    long long flops     = total;          // 1 add per element
    long long g_reads   = 2 * total;      // read A and B
    long long g_writes  = total;          // write C

    printf("===== Kernel Analysis =====\n");
    printf("Total Elements:       %lld\n", total);
    printf("Floating Ops (FLOPs): %lld\n", flops);
    printf("Global Reads:         %lld\n", g_reads);
    printf("Global Writes:        %lld\n", g_writes);
    printf("Speedup:              %.2fx\n\n", cpu_time / gpu_time);

    // Show sample results
    printf("===== Sample Results =====\n");
    printf("%-10s %-10s %-10s %-10s\n",
           "Index", "A[i]", "B[i]", "C[i]");
    for (int i = 0; i < 5; i++) {
        printf("%-10d %-10d %-10d %-10d\n",
               i, h_A[i], h_B[i], h_C[i]);
    }

    // Free everything
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    free(h_Cref);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}

Writing matrix_add.cu


In [11]:
!nvcc matrix_add.cu -o matrix_add && ./matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
========== Matrix Addition ==========
Matrix Size: 2048 x 2048

CPU Time: 30.7500 ms
Block Dim: 32 x 32 = 1024 threads/block
Grid  Dim: 64 x 64 = 4096 blocks

GPU Time: 17.6912 ms

Verification: PASS ✓ (errors: 0)

===== Kernel Analysis =====
Total Elements:       4194304
Floating Ops (FLOPs): 4194304
Global Reads:         8388608
Global Writes:        4194304
Speedup:              1.74x

===== Sample Results =====
Index      A[i]       B[i]       C[i]      
0          166        740        906       
1          881        241        1122      
2          12         758        770       
3          21         940        961       
4          535        743        1278      
